# 05 · Objects, classes, and exceptions · Worked solutions

Try the exercises in the student chapter first. This notebook is self-contained: run it from
a fresh kernel in order. Each exercise explains one implementation and then tests both the
normal case and cases that reveal common mistakes. Assertions here are checks of our own
code; required user-input validation is implemented with explicit exceptions.


## Exercise 1 · Independent task lists

`None` avoids a shared default list, and `list(tasks)` isolates the stored collection from
the caller's collection. The method's `__self__` is the instance whose tasks it will modify.
The copy is shallow; strings are sufficient here because they are immutable.


In [ ]:
class TaskList:
    def __init__(self, owner, tasks=None):
        self.owner = owner
        self.tasks = [] if tasks is None else list(tasks)

    def add_task(self, text):
        self.tasks.append(text)

    def __repr__(self):
        return f"TaskList(owner={self.owner!r}, tasks={self.tasks!r})"

first = TaskList("Ada")
second = TaskList("Lin")
first.add_task("read")
assert first.tasks == ["read"]
assert second.tasks == []
assert first.tasks is not second.tasks
original = ["practice"]
copied = TaskList("Bo", original)
original.append("rest")
assert copied.tasks == ["practice"]
assert first.add_task.__self__ is first
assert first.add_task.__func__ is TaskList.add_task
assert "Ada" in repr(first) and "read" in repr(first)
print(first, second)


## Exercise 2 · Points and mathematical operations

Addition creates a new result; rotation deliberately changes the receiver. Returning
`NotImplemented` gives Python a chance to resolve an unsupported binary operation. Supplying
a `Point()` start value also makes the empty `sum` result a point. `hypot` calculates the
Euclidean distance without manually squaring and taking a square root.


In [ ]:
from math import hypot

class Point:
    """A mutable two-dimensional point with addition and coordinate iteration."""

    def __init__(self, x=0, y=0):
        self.x = x
        self.y = y

    def rotate_90_ccw(self):
        """Rotate this point 90 degrees counterclockwise around the origin."""
        self.x, self.y = -self.y, self.x

    def distance_to(self, other):
        if not isinstance(other, Point):
            raise TypeError("other must be a Point")
        return hypot(self.x - other.x, self.y - other.y)

    def __add__(self, other):
        if not isinstance(other, Point):
            return NotImplemented
        return Point(self.x + other.x, self.y + other.y)

    def __eq__(self, other):
        if not isinstance(other, Point):
            return NotImplemented
        return (self.x, self.y) == (other.x, other.y)

    def __iter__(self):
        return iter((self.x, self.y))

    def __repr__(self):
        return f"Point({self.x!r}, {self.y!r})"

    def __str__(self):
        return f"({self.x}, {self.y})"

origin = Point()
point = Point(3, 4)
assert origin.distance_to(point) == 5.0
assert point.distance_to(point) == 0.0
for _ in range(4):
    point.rotate_90_ccw()
assert point == Point(3, 4)
other = Point(-3, 2)
combined = point + other
assert combined == Point(0, 6)
assert point == Point(3, 4) and other == Point(-3, 2)
assert combined is not point and combined is not other
assert sum([point, other], start=Point()) == Point(0, 6)
assert sum([], start=Point()) == Point()
assert Point.__add__(point, 3) is NotImplemented
assert point != (3, 4)
for operation in [lambda: point + 3, lambda: point.distance_to((3, 4))]:
    try:
        operation()
    except TypeError:
        pass
    else:
        raise AssertionError("Unsupported argument should raise TypeError")
print(repr(combined), origin.distance_to(point))


## Exercise 3 · Single and multiple inheritance

For a `Diamond`, `super()` inside `Left` continues to `Right` because `Right` comes after
`Left` in that instance's MRO. Each override delegates once, so `Root` appears once.
This cooperative chain depends on compatible signatures and deliberate use of `super`.


In [ ]:
class Course:
    """A base class whose method a subclass can extend."""

    def __init__(self, title):
        self.title = title

    def describe(self):
        return self.title


class OnlineCourse(Course):
    def __init__(self, title, platform):
        super().__init__(title)
        self.platform = platform

    def describe(self):
        return f"{super().describe()} on {self.platform}"

course = OnlineCourse("Python", "Jupyter")
assert course.title == "Python"
assert course.describe() == "Python on Jupyter"
assert isinstance(course, Course)
assert issubclass(OnlineCourse, Course)

class Root:
    def labels(self):
        return ["Root"]

class Left(Root):
    def labels(self):
        return ["Left"] + super().labels()

class Right(Root):
    def labels(self):
        return ["Right"] + super().labels()

class Diamond(Left, Right):
    pass

assert Diamond().labels() == ["Left", "Right", "Root"]
assert [base.__name__ for base in Diamond.mro()] == [
    "Diamond", "Left", "Right", "Root", "object"
]
assert Left().labels() == ["Left", "Root"]
assert Right().labels() == ["Right", "Root"]
print(course.describe(), Diamond().labels())


## Exercise 4 · A container with fresh iterators

The internal list handles standard indexing and slicing behavior, including exceptions.
Returning `iter(self._titles)` supplies a new iterator each time, allowing both repeated
traversal and two traversals at different positions. Returning `self` would only make sense
if the object implemented iterator state and `__next__`, which this reusable container does not.


In [ ]:
class ReadingList:
    """A small container that delegates its operations to a private list."""

    def __init__(self, titles=()):
        self._titles = list(titles)

    def __len__(self):
        return len(self._titles)

    def __contains__(self, title):
        return title in self._titles

    def __getitem__(self, index):
        return self._titles[index]

    def __iter__(self):
        return iter(self._titles)

titles = ["Python", "Data", "Design"]
shelf = ReadingList(titles)
titles.append("Changed later")
assert len(shelf) == 3
assert shelf[-1] == "Design"
assert shelf[1:] == ["Data", "Design"]
assert "Data" in shelf and "Unknown" not in shelf
assert list(shelf) == list(shelf) == ["Python", "Data", "Design"]
left, right = iter(shelf), iter(shelf)
assert next(left) == "Python" and next(left) == "Data"
assert next(right) == "Python"
empty = ReadingList()
assert len(empty) == 0 and list(empty) == [] and "Python" not in empty
for container, index in [(shelf, 3), (empty, 0)]:
    try:
        container[index]
    except IndexError:
        pass
    else:
        raise AssertionError("An invalid index must raise IndexError")
print(list(shelf))


## Exercise 5 · Preserve state on a rejected booking

All validation happens before the assignment that changes `booked`. `CapacityError` is
separate from a wrong input type or invalid numeric value, allowing callers to distinguish
those situations. Booleans are rejected explicitly because `isinstance(True, int)` is true.


In [ ]:
class CapacityError(Exception):
    """A booking would exceed the available capacity."""


class Workshop:
    """Keep bookings between zero and a fixed nonnegative integer capacity."""

    def __init__(self, capacity):
        if isinstance(capacity, bool) or not isinstance(capacity, int):
            raise TypeError("capacity must be an integer")
        if capacity < 0:
            raise ValueError("capacity must not be negative")
        self.capacity = capacity
        self.booked = 0

    def book(self, places=1):
        if isinstance(places, bool) or not isinstance(places, int):
            raise TypeError("places must be an integer")
        if places <= 0:
            raise ValueError("places must be positive")
        if self.booked + places > self.capacity:
            raise CapacityError("not enough places available")
        self.booked += places
        return self.capacity - self.booked

workshop = Workshop(3)
assert workshop.book(2) == 1
assert workshop.book() == 0
assert workshop.booked == 3
for places, expected in [(1, CapacityError), (0, ValueError), (-1, ValueError),
                         (1.5, TypeError), (True, TypeError)]:
    previous = workshop.booked
    try:
        workshop.book(places)
    except expected:
        pass
    else:
        raise AssertionError(f"Expected {expected.__name__}")
    assert workshop.booked == previous

empty = Workshop(0)
try:
    empty.book()
except CapacityError:
    pass
else:
    raise AssertionError("A zero-capacity workshop cannot accept a booking")
assert empty.booked == 0
for capacity, expected in [(-1, ValueError), (2.5, TypeError), (False, TypeError)]:
    try:
        Workshop(capacity)
    except expected:
        pass
    else:
        raise AssertionError(f"Expected {expected.__name__}")
print("All workshop states stayed valid.")


## Exercise 6 · Separate parsing, success, and cleanup

The context manager owns closing the stream. The inner `finally` records its event before
the surrounding `with` closes the stream. An invalid value is recorded and re-raised;
an unrelated failure is not misclassified as invalid text. Even then `finally` runs and
the context manager closes the resource before the caller receives the exception.


In [ ]:
from io import StringIO

def read_integer(stream, events):
    with stream as active:
        try:
            value = int(active.read())
        except ValueError:
            events.append("invalid")
            raise
        else:
            events.append("parsed")
            return value
        finally:
            events.append("cleanup")

events = []
valid = StringIO(" 42\n")
assert read_integer(valid, events) == 42
assert events == ["parsed", "cleanup"]
assert valid.closed

events = []
invalid = StringIO("forty-two")
try:
    read_integer(invalid, events)
except ValueError:
    pass
else:
    raise AssertionError("Invalid text should propagate ValueError")
assert events == ["invalid", "cleanup"]
assert invalid.closed

class BrokenStream(StringIO):
    def read(self, *args, **kwargs):
        raise OSError("simulated read failure")

events = []
broken = BrokenStream("42")
try:
    read_integer(broken, events)
except OSError:
    pass
else:
    raise AssertionError("Unexpected I/O errors must propagate")
assert events == ["cleanup"]
assert broken.closed
print("Success, invalid text, and unexpected failure all clean up.")
